# PUSHUP POSTURE CLASSIFIER - TRAINING & EVALUATION
Script ini digunakan untuk melatih Multi-Layer Perceptron (MLP) Neural Network menggunakan ekstraksi fitur dari MediaPipe (disimpan di `landmarks.npz`). Script ini juga menampilkan metrik evaluasi lengkap untuk kebutuhan laporan UAS.

In [ ]:
# Install library (jika dijalankan di environment kosong)
!pip install scikit-learn pandas matplotlib seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, classification_report
)

# =========================================
# 1. LOAD DATA
# =========================================
# Pastikan Anda sudah mengupload file 'landmarks.npz' ke Colab
features_path = "landmarks.npz"

if not os.path.exists(features_path):
    print(f"File {features_path} tidak ditemukan! Silakan upload dulu.")
else:
    data = np.load(features_path)
    X, y = data["X"], data["y"]
    print(f"Dataset berhasil dimuat!")
    print(f"Total Samples : {X.shape[0]}")
    print(f"Feature Dim   : {X.shape[1]}")
    print(f"Class 1 (Correct) : {(y==1).sum()}")
    print(f"Class 0 (Wrong)   : {(y==0).sum()}")

    # =========================================
    # 2. SPLIT & SCALING DATA
    # =========================================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # =========================================
    # 3. TRAINING MODEL
    # =========================================
    clf = MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation="relu",
        solver="adam",
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        verbose=False
    )

    print("\nMelatih MLP Classifier...")
    clf.fit(X_train_scaled, y_train)
    print("Training Selesai!")

    # =========================================
    # 4. EVALUASI METRIK
    # =========================================
    y_pred = clf.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Membuat Tabel Metrik (Bagus untuk screenshot laporan UAS)
    metrics_df = pd.DataFrame([{
        "Accuracy": f"{acc*100:.2f}%",
        "Precision": f"{prec*100:.2f}%",
        "Recall": f"{rec*100:.2f}%",
        "F1-Score": f"{f1*100:.2f}%"
    }])
    
    print("\n=== HASIL EVALUASI MODEL ===")
    display(metrics_df)

    print("\nClassification Report Detailed:")
    print(classification_report(y_test, y_pred, target_names=["Wrong (0)", "Correct (1)"]))

    # =========================================
    # 5. VISUALISASI CONFUSION MATRIX
    # =========================================
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Wrong (0)', 'Correct (1)'], 
                yticklabels=['Wrong (0)', 'Correct (1)'],
                annot_kws={"size": 14})
    plt.title('Confusion Matrix - Pushup Classification', fontsize=16)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.show()

    # =========================================
    # 6. SAVE MODEL (untuk di-download)
    # =========================================
    joblib.dump(clf, "pushup_classifier.pkl")
    joblib.dump(scaler, "scaler.pkl")
    print("\nModel dan Scaler berhasil disimpan sebagai .pkl!")
    print("Silakan download file tersebut untuk dipakai di Backend FastAPI.")